In [13]:
import pandas as pd
import os

ruta = r'C:\Users\Emdro\OneDrive\Escritorio\Equipo_04\FalconChallenge\data'

def cargar_ibwc_robusto(nombre_archivo, nombre_valor):
    ruta_completa = os.path.join(ruta, nombre_archivo)
    df_raw = pd.read_csv(ruta_completa, header=None, names=['col1', 'col2'],
                          skip_blank_lines=True, on_bad_lines='skip')
    
    fechas_parseadas = pd.to_datetime(df_raw['col1'], errors='coerce', format='mixed')
    mask_validas = fechas_parseadas.notna()
    
    df_limpio = df_raw[mask_validas].copy()
    df_limpio['Timestamp'] = fechas_parseadas[mask_validas]
    df_limpio[nombre_valor] = pd.to_numeric(df_limpio['col2'], errors='coerce')
    
    df_limpio = df_limpio[['Timestamp', nombre_valor]].dropna()
    df_limpio = df_limpio.set_index('Timestamp').sort_index()
    
    print(f"{nombre_archivo[:45]}... con {len(df_limpio)} filas | "
          f"{df_limpio.index.min()} a {df_limpio.index.max()}")
    return df_limpio

archivos = {
    'discharge':  'DataSetExport-Discharge.Best Available@08461300-Instantaneous-m^3 s-20260629185451.csv',
    'lake_area':  'DataSetExport-Lake Area.Best Available@08461200-Instantaneous-m^2-20260629185344.csv',
    'elevation':  'DataSetExport-Reservoir Elevation.Web-Daily-m@08461200-Instantaneous-m-20260629185508.csv',
    'storage':    'DataSetExport-Total Storage.Web-Daily-tcm@08461200-Instantaneous-m^3-20260629185416.csv',
}

df_discharge = cargar_ibwc_robusto(archivos['discharge'], 'Discharge_m3s')
df_lake_area = cargar_ibwc_robusto(archivos['lake_area'], 'LakeArea_m2')
df_elevation = cargar_ibwc_robusto(archivos['elevation'], 'Elevation_m')
df_storage   = cargar_ibwc_robusto(archivos['storage'],   'Storage_m3')

DataSetExport-Discharge.Best Available@084613... con 35023 filas | 2025-06-29 00:00:00 a 2026-06-29 00:00:00
DataSetExport-Lake Area.Best Available@084612... con 35027 filas | 2025-06-29 00:00:00 a 2026-06-29 00:00:00
DataSetExport-Reservoir Elevation.Web-Daily-m... con 366 filas | 2025-06-29 00:00:00 a 2026-06-29 00:00:00
DataSetExport-Total Storage.Web-Daily-tcm@084... con 366 filas | 2025-06-29 00:00:00 a 2026-06-29 00:00:00


In [14]:
# ── Discharge: instantáneo cada 15 min → volumen semanal ──
discharge_diario = df_discharge['Discharge_m3s'].resample('D').mean()  # m³/s promedio diario
discharge_vol_diario = discharge_diario * 86400  # m³/día
R_obs_semanal = discharge_vol_diario.resample('W').sum()  # m³/semana

# ── Storage: a semanal y delta ──
storage_semanal = df_storage['Storage_m3'].resample('W').mean()
delta_S_obs_semanal = storage_semanal.diff().dropna()

# ── Alinear fechas comunes ──
fechas_comunes = R_obs_semanal.index.intersection(delta_S_obs_semanal.index)
R_obs_final        = R_obs_semanal.loc[fechas_comunes]
delta_S_obs_final  = delta_S_obs_semanal.loc[fechas_comunes]

S0_real    = storage_semanal.loc[fechas_comunes[0]]
Smax_real  = storage_semanal.max()
Smin_real  = 0.25 * Smax_real

print(f"\nSemanas disponibles: {len(fechas_comunes)}")
print(f"S0    = {S0_real:,.0f} m³")
print(f"Smax  = {Smax_real:,.0f} m³")
print(f"Smin  = {Smin_real:,.0f} m³")
print(f"\nR_obs (primeras 5 semanas):")
print(R_obs_final.head())
print(f"\nΔS_obs (primeras 5 semanas):")
print(delta_S_obs_final.head())


Semanas disponibles: 53
S0    = 378,683,545 m³
Smax  = 648,840,000 m³
Smin  = 162,210,000 m³

R_obs (primeras 5 semanas):
Timestamp
2025-07-06    3.290567e+06
2025-07-13    8.086461e+06
2025-07-20    7.562386e+06
2025-07-27    8.351164e+06
2025-08-03    1.511289e+07
Freq: W-SUN, Name: Discharge_m3s, dtype: float64

ΔS_obs (primeras 5 semanas):
Timestamp
2025-07-06    3.072321e+06
2025-07-13    3.217460e+06
2025-07-20    5.183217e+06
2025-07-27    1.516312e+07
2025-08-03   -2.218838e+06
Freq: W-SUN, Name: Storage_m3, dtype: float64


In [8]:
import os

ruta = r'C:\Users\Emdro\OneDrive\Escritorio\Equipo_04\FalconChallenge\data'
archivo = 'DataSetExport-Discharge.Best Available@08461300-Instantaneous-m^3 s-20260629185451.csv'

with open(os.path.join(ruta, archivo), encoding='utf-8') as f:
    for i in range(8):
        linea = f.readline()
        print(f"[{i}] {linea[:120]}")

[0] #Data Set Export - Discharge.Best Available@08461300,

[1] Timestamp (UTC-06:00),Value (m^3/s)

[2] 2025-06-29 00:00:00,11.9892907608718

[3] 2025-06-29 00:15:00,12.037480638939972

[4] 2025-06-29 00:30:00,12.033507482363452

[5] 2025-06-29 00:45:00,11.977396732936027

[6] 2025-06-29 01:00:00,12.025563105383736

[7] 2025-06-29 01:15:00,12.021591884987311



In [15]:
import numpy as np

# ── Parámetros oficiales con datos reales ──────────
T_total = len(R_obs_final)  # 53 semanas disponibles
L = 5

R_obs = R_obs_final.values
delta_S_obs = delta_S_obs_final.values
S0 = S0_real
Smax = Smax_real
Smin = Smin_real

# Delta u oficial: 0.25 * mediana semanal de R_obs
delta_u = 0.25 * np.median(R_obs)
umax = 2 * delta_u
eta = 0.10

nivel_vals = np.array([-2, -1, 0, 1, 2]) * delta_u

print(f"delta_u = {delta_u:,.0f} m³")
print(f"umax    = {umax:,.0f} m³")
print(f"Niveles de ajuste: {nivel_vals}")
print(f"\nS0   = {S0:,.0f} m³")
print(f"Smax = {Smax:,.0f} m³")
print(f"Smin = {Smin:,.0f} m³")

delta_u = 2,953,742 m³
umax    = 5,907,485 m³
Niveles de ajuste: [-5907484.6795905  -2953742.33979525        0.          2953742.33979525
  5907484.6795905 ]

S0   = 378,683,545 m³
Smax = 648,840,000 m³
Smin = 162,210,000 m³


In [16]:
# Pesos oficiales para T=26 para la instancia mediana.
T_oficial = 26

w1 = 1.0 / ((T_oficial + 1) * Smin**2)
w2 = 0.1 / (T_oficial * umax**2)
w3 = 0.1 / ((T_oficial - 1) * (2*umax)**2)

print(f"w1 = {w1:.4e}")
print(f"w2 = {w2:.4e}")
print(f"w3 = {w3:.4e}")

w1 = 1.4076e-18
w2 = 1.1021e-16
w3 = 2.8655e-17


In [17]:
def idx(t, l, L=5):
    return t * L + l

def Q_crit(T, L, nivel_vals, A, Smin):
    N = T * L
    Q = np.zeros((N, N))
    B = Smin - A
    for t in range(1, T + 1):
        coef = np.zeros(N)
        for k in range(t):
            for l in range(L):
                coef[idx(k, l, L)] += nivel_vals[l]
        for i in range(N):
            Q[i, i] += 2 * B[t] * coef[i]
            for j in range(i, N):
                if i == j:
                    Q[i, i] += coef[i]**2
                else:
                    Q[i, j] += 2 * coef[i] * coef[j]
    return Q

def Q_dev(T, L, nivel_vals):
    N = T * L
    Q = np.zeros((N, N))
    for t in range(T):
        for l in range(L):
            for l2 in range(l, L):
                i, j = idx(t, l, L), idx(t, l2, L)
                if i == j:
                    Q[i, i] += nivel_vals[l]**2
                else:
                    Q[i, j] += 2 * nivel_vals[l] * nivel_vals[l2]
    return Q

def Q_smooth(T, L, nivel_vals):
    N = T * L
    Q = np.zeros((N, N))
    for t in range(1, T):
        for l in range(L):
            for l2 in range(L):
                i, j = idx(t, l, L), idx(t-1, l2, L)
                r, c = min(i, j), max(i, j)
                Q[r, c] += (nivel_vals[l] - nivel_vals[l2])**2
    return Q

def Q_onehot(T, L):
    N = T * L
    Q = np.zeros((N, N))
    for t in range(T):
        for l in range(L):
            i = idx(t, l, L)
            Q[i, i] -= 1
        for l in range(L):
            for l2 in range(l+1, L):
                i, j = idx(t, l, L), idx(t, l2, L)
                Q[i, j] += 2
    return Q

# ── CASO T=1 con primera semana real ──────────────
T1 = 1
A1 = np.zeros(T1 + 1)
A1[0] = S0
A1[1] = A1[0] + delta_S_obs[0]

Qc1 = Q_crit(T1, L, nivel_vals, A1, Smin)
Qd1 = Q_dev(T1, L, nivel_vals)
Qs1 = Q_smooth(T1, L, nivel_vals)  # debería ser todo ceros (no hay t-1)
Qo1 = Q_onehot(T1, L)

print("Q_smooth para T=1 (debe ser todo ceros):")
print(Qs1)
print(f"\nSuma absoluta: {np.abs(Qs1).sum()}  (debe ser 0)")

Q_smooth para T=1 (debe ser todo ceros):
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

Suma absoluta: 0.0  (debe ser 0)


In [18]:
lam = 1e10 * max(w1, w2, w3)  # lambda suficientemente grande relativo a la escala real
Q_total_1 = w1*Qc1 + w2*Qd1 + w3*Qs1 + lam*Qo1

# Fuerza bruta: solo 5 combinaciones posibles para T=1
mejor_e = np.inf
mejor_x = None
for l in range(L):
    x = np.zeros(L)
    x[l] = 1
    e = x @ Q_total_1 @ x
    print(f"u={nivel_vals[l]:>15,.0f} (nivel {l}) → energía = {e:.6e}")
    if e < mejor_e:
        mejor_e = e
        mejor_x = x

print(f"\nMejor solución: nivel {np.argmax(mejor_x)}, u={nivel_vals[np.argmax(mejor_x)]:,.0f}")

u=     -5,907,485 (nivel 0) → energía = 7.545402e-03
u=     -2,953,742 (nivel 1) → energía = 2.798331e-03
u=              0 (nivel 2) → energía = -1.102101e-06
u=      2,953,742 (nivel 3) → energía = -8.528964e-04
u=      5,907,485 (nivel 4) → energía = 2.429477e-04

Mejor solución: nivel 3, u=2,953,742


In [22]:
T2 = 2
A2 = np.zeros(T2 + 1)
A2[0] = S0
A2[1] = A2[0] + delta_S_obs[0]
A2[2] = A2[1] + delta_S_obs[1]

Qc2 = Q_crit(T2, L, nivel_vals, A2, Smin)
Qd2 = Q_dev(T2, L, nivel_vals)
Qs2 = Q_smooth(T2, L, nivel_vals)
Qo2 = Q_onehot(T2, L)

print("Q_smooth para T=2:")
print(f"Suma absoluta: {np.abs(Qs2).sum():.4e}")

lam2 = 1e10 * max(w1, w2, w3)
Q_total_2 = w1*Qc2 + w2*Qd2 + w3*Qs2 + lam2*Qo2

# Fuerza bruta: 5*5 = 25 combinaciones
from itertools import product
mejor_e = np.inf
mejor_sol = None

for l0, l1 in product(range(L), repeat=2):
    x = np.zeros(T2*L)
    x[idx(0, l0, L)] = 1
    x[idx(1, l1, L)] = 1
    e = x @ Q_total_2 @ x
    if e < mejor_e:
        mejor_e = e
        mejor_sol = (l0, l1)

print(f"\nMejor solución T=2: u(0)={nivel_vals[mejor_sol[0]]:,.0f}, "
      f"u(1)={nivel_vals[mejor_sol[1]]:,.0f}")
print(f"Energía: {mejor_e:.6e}")

Q_smooth para T=2:
Suma absoluta: 8.7246e+14

Mejor solución T=2: u(0)=5,907,485, u(1)=2,953,742
Energía: -3.993193e-03


In [24]:
# Verificar contribución de cada término por separado en la solución óptima
x_opt = np.zeros(T2*L)
x_opt[idx(0, 4, L)] = 1  # nivel 4 = +5,907,485
x_opt[idx(1, 3, L)] = 1  # nivel 3 = +2,953,742

e_crit = w1 * (x_opt @ Qc2 @ x_opt)
e_dev  = w2 * (x_opt @ Qd2 @ x_opt)
e_smooth = w3 * (x_opt @ Qs2 @ x_opt)
e_onehot = lam2 * (x_opt @ Qo2 @ x_opt)

print(f"Contribución Ccrit:    {e_crit:.6e}")
print(f"Contribución Cdev:     {e_dev:.6e}")
print(f"Contribución Csmooth:  {e_smooth:.6e}")
print(f"Contribución onehot:   {e_onehot:.6e}")
print(f"Suma total:            {e_crit+e_dev+e_smooth+e_onehot:.6e}")

Contribución Ccrit:    -9.048681e-03
Contribución Cdev:     4.807692e-03
Contribución Csmooth:  2.500000e-04
Contribución onehot:   -2.204202e-06
Suma total:            -3.993193e-03


In [25]:
T3 = 3
A3 = np.zeros(T3 + 1)
A3[0] = S0
for t in range(T3):
    A3[t+1] = A3[t] + delta_S_obs[t]

Qc3 = Q_crit(T3, L, nivel_vals, A3, Smin)
Qd3 = Q_dev(T3, L, nivel_vals)
Qs3 = Q_smooth(T3, L, nivel_vals)
Qo3 = Q_onehot(T3, L)

lam3 = 1e10 * max(w1, w2, w3)
Q_total_3 = w1*Qc3 + w2*Qd3 + w3*Qs3 + lam3*Qo3

# Fuerza bruta: 5^3 = 125 combinaciones, todavía manejable
from itertools import product

mejor_e = np.inf
mejor_sol = None

for l0, l1, l2 in product(range(L), repeat=3):
    x = np.zeros(T3*L)
    x[idx(0, l0, L)] = 1
    x[idx(1, l1, L)] = 1
    x[idx(2, l2, L)] = 1
    e = x @ Q_total_3 @ x
    if e < mejor_e:
        mejor_e = e
        mejor_sol = (l0, l1, l2)

print(f"Mejor solución T=3: u(0)={nivel_vals[mejor_sol[0]]:,.0f}, "
      f"u(1)={nivel_vals[mejor_sol[1]]:,.0f}, u(2)={nivel_vals[mejor_sol[2]]:,.0f}")
print(f"Energía: {mejor_e:.6e}")

# Verificar factibilidad física básica (fuera del QUBO)
u_sol = np.array([nivel_vals[l] for l in mejor_sol])
S_sol = np.zeros(T3+1)
S_sol[0] = S0
for t in range(T3):
    S_sol[t+1] = S_sol[t] + delta_S_obs[t] - u_sol[t]

print(f"\nTrayectoria S(t): {S_sol}")
print(f"¿Algún S(t) < Smin? {np.any(S_sol[1:] < Smin)}")
print(f"¿Algún S(t) > Smax? {np.any(S_sol[1:] > Smax)}")
print(f"¿Algún R(t) negativo? {np.any(R_obs[:T3] + u_sol < 0)}")

Mejor solución T=3: u(0)=5,907,485, u(1)=5,907,485, u(2)=2,953,742
Energía: -1.108487e-02

Trayectoria S(t): [3.78683545e+08 3.75848382e+08 3.73158357e+08 3.75387831e+08]
¿Algún S(t) < Smin? False
¿Algún S(t) > Smax? False
¿Algún R(t) negativo? False


## Simulación de prueba.

In [26]:
import time

# ── Construir QUBO completo para T=26 ──────────────
T_oficial = 26

A26 = np.zeros(T_oficial + 1)
A26[0] = S0
for t in range(T_oficial):
    A26[t+1] = A26[t] + delta_S_obs[t]

Qc26 = Q_crit(T_oficial, L, nivel_vals, A26, Smin)
Qd26 = Q_dev(T_oficial, L, nivel_vals)
Qs26 = Q_smooth(T_oficial, L, nivel_vals)
Qo26 = Q_onehot(T_oficial, L)

lam26 = 1e10 * max(w1, w2, w3)
Q_total_26 = w1*Qc26 + w2*Qd26 + w3*Qs26 + lam26*Qo26

print(f"Tamaño del QUBO: {Q_total_26.shape}")
print(f"Variables: {T_oficial * L}")

Tamaño del QUBO: (130, 130)
Variables: 130


In [27]:
# ── Simulated Annealing sobre la matriz QUBO ───────
def evaluar_factible(x, T, L, nivel_vals, R_obs, S0, delta_S_obs,
                      Smax, eta):
    """Filtrado de factibilidad clásico, fuera del QUBO"""
    u_seq = np.array([nivel_vals[np.argmax(x[t*L:(t+1)*L])] for t in range(T)])
    
    if np.any(R_obs[:T] + u_seq < 0):
        return False, None
    if abs(np.sum(u_seq)) > eta * np.sum(R_obs[:T]):
        return False, None
    
    S = np.zeros(T+1)
    S[0] = S0
    for t in range(T):
        S[t+1] = S[t] + delta_S_obs[t] - u_seq[t]
        if S[t+1] < 0 or S[t+1] > Smax:
            return False, None
    
    return True, (u_seq, S)

def simulated_annealing_qubo(Q, T, L, n_iter=60000, T0=1.0, Tf=1e-4):
    N = T * L
    alpha = (Tf/T0)**(1/n_iter)
    
    # Inicializar con one-hot válido (nivel neutro=índice 2 en cada t)
    x = np.zeros(N)
    for t in range(T):
        x[idx(t, 2, L)] = 1  # arranca con u=0 en todas las semanas
    
    e_actual = x @ Q @ x
    x_mejor = x.copy()
    e_mejor = e_actual
    temp = T0
    
    log = []
    t_inicio = time.time()
    
    for i in range(n_iter):
        x_nuevo = x.copy()
        # Perturbación: cambia el nivel activo de una semana aleatoria
        t_cambiar = np.random.randint(T)
        for l in range(L):
            x_nuevo[idx(t_cambiar, l, L)] = 0
        l_nuevo = np.random.randint(L)
        x_nuevo[idx(t_cambiar, l_nuevo, L)] = 1
        
        e_nuevo = x_nuevo @ Q @ x_nuevo
        delta = e_nuevo - e_actual
        
        if delta < 0 or np.random.rand() < np.exp(-delta/temp):
            x = x_nuevo
            e_actual = e_nuevo
            if e_actual < e_mejor:
                x_mejor = x.copy()
                e_mejor = e_actual
        
        temp *= alpha
        if i % 5000 == 0:
            log.append(e_mejor)
    
    tiempo = time.time() - t_inicio
    return x_mejor, e_mejor, log, tiempo

print("Resolviendo T=26 con Simulated Annealing...")
x_opt, e_opt, log_sa, tiempo_sa = simulated_annealing_qubo(Q_total_26, T_oficial, L)
print(f"Tiempo: {tiempo_sa:.2f}s | Energía final: {e_opt:.6e}")

Resolviendo T=26 con Simulated Annealing...
Tiempo: 2.16s | Energía final: -7.436297e-01


In [28]:
# ── Extraer solución y verificar factibilidad ──────
factible, datos = evaluar_factible(x_opt, T_oficial, L, nivel_vals,
                                     R_obs, S0, delta_S_obs, Smax, eta)
print(f"¿Solución factible? {factible}")

if factible:
    u_qubo, S_qubo = datos
    print(f"\nSecuencia u(t):\n{u_qubo}")
    print(f"\nTrayectoria S(t):\n{S_qubo}")

¿Solución factible? False


In [29]:
def calcular_SRS(u_seq, S, w1, w2, w3, Smin):
    Ccrit   = np.sum(np.maximum(0, Smin - S)**2)
    Cdev    = np.sum(u_seq**2)
    Csmooth = np.sum(np.diff(u_seq)**2)
    return -(w1*Ccrit + w2*Cdev + w3*Csmooth)

# Baseline histórico
u_hist = np.zeros(T_oficial)
S_hist = np.zeros(T_oficial+1)
S_hist[0] = S0
for t in range(T_oficial):
    S_hist[t+1] = S_hist[t] + delta_S_obs[t]
SRS_hist = calcular_SRS(u_hist, S_hist, w1, w2, w3, Smin)

# Baseline regla de umbral
u_rule = np.zeros(T_oficial)
S_rule = np.zeros(T_oficial+1)
S_rule[0] = S0
for t in range(T_oficial):
    u_rule[t] = -delta_u if S_rule[t] < Smin else 0.0
    S_rule[t+1] = S_rule[t] + delta_S_obs[t] - u_rule[t]
SRS_rule = calcular_SRS(u_rule, S_rule, w1, w2, w3, Smin)

# QUBO/SA
SRS_qubo = calcular_SRS(u_qubo, S_qubo, w1, w2, w3, Smin) if factible else None

print(f"SRS histórico:  {SRS_hist:.6e}")
print(f"SRS umbral:     {SRS_rule:.6e}")
print(f"SRS QUBO/SA:    {SRS_qubo:.6e}" if factible else "QUBO no factible — ajustar penalización")
print(f"\nΔSRS vs histórico: {SRS_qubo - SRS_hist:.6e}" if factible else "")
print(f"ΔSRS vs umbral:    {SRS_qubo - SRS_rule:.6e}" if factible else "")

SRS histórico:  -0.000000e+00
SRS umbral:     -0.000000e+00
QUBO no factible — ajustar penalización




In [ ]:
def simulated_annealing_qubo_factible(Q, T, L, nivel_vals, R_obs, S0,
                                        delta_S_obs, Smax, eta,
                                        n_iter=80000, T0=1.0, Tf=1e-4):
    N = T * L
    alpha = (Tf/T0)**(1/n_iter)

    def extraer_u(x):
        return np.array([nivel_vals[np.argmax(x[t*L:(t+1)*L])] for t in range(T)])

    def es_factible(u_seq):
        if np.any(R_obs[:T] + u_seq < 0):
            return False
        if abs(np.sum(u_seq)) > eta * np.sum(R_obs[:T]):
            return False
        S = np.zeros(T+1)
        S[0] = S0
        for t in range(T):
            S[t+1] = S[t] + delta_S_obs[t] - u_seq[t]
            if S[t+1] < 0 or S[t+1] > Smax:
                return False
        return True

    # Arrancar con u=0 en todas las semanas (siempre factible, reproduce histórico)
    x = np.zeros(N)
    for t in range(T):
        x[idx(t, 2, L)] = 1
    
    u_actual = extraer_u(x)
    assert es_factible(u_actual), 

    e_actual = x @ Q @ x
    x_mejor, e_mejor = x.copy(), e_actual
    temp = T0
    log = []
    rechazos_infactibles = 0

    t_inicio = time.time()
    for i in range(n_iter):
        x_nuevo = x.copy()
        t_cambiar = np.random.randint(T)
        for l in range(L):
            x_nuevo[idx(t_cambiar, l, L)] = 0
        l_nuevo = np.random.randint(L)
        x_nuevo[idx(t_cambiar, l_nuevo, L)] = 1

        u_nuevo = extraer_u(x_nuevo)
        
        # ── FILTRO DE FACTIBILIDAD: rechaza el movimiento si rompe restricciones ──
        if not es_factible(u_nuevo):
            rechazos_infactibles += 1
            continue  # no acepta este movimiento, sigue con el actual

        e_nuevo = x_nuevo @ Q @ x_nuevo
        delta = e_nuevo - e_actual

        if delta < 0 or np.random.rand() < np.exp(-delta/temp):
            x = x_nuevo
            e_actual = e_nuevo
            if e_actual < e_mejor:
                x_mejor, e_mejor = x.copy(), e_actual

        temp *= alpha
        if i % 8000 == 0:
            log.append(e_mejor)

    tiempo = time.time() - t_inicio
    print(f"Movimientos rechazados por infactibilidad: {rechazos_infactibles}/{n_iter}")
    return x_mejor, e_mejor, log, tiempo


print("Resolviendo T=26 con SA + filtrado de factibilidad...")
x_opt, e_opt, log_sa, tiempo_sa = simulated_annealing_qubo_factible(
    Q_total_26, T_oficial, L, nivel_vals, R_obs, S0, delta_S_obs, Smax, eta
)
print(f"Tiempo: {tiempo_sa:.2f}s | Energía final: {e_opt:.6e}")

factible, datos = evaluar_factible(x_opt, T_oficial, L, nivel_vals,
                                     R_obs, S0, delta_S_obs, Smax, eta)
print(f"¿Solución factible? {factible}")

Resolviendo T=26 con SA + filtrado de factibilidad...
Movimientos rechazados por infactibilidad: 16608/80000
Tiempo: 9.50s | Energía final: -5.236622e-01
¿Solución factible? True
